# 09 — Publication figures and tables

Run this last. It creates lightweight paper-ready figures from formal validation,
held-out test, LOIO, and transfer-analysis outputs.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)

In [ ]:
from carnivore_reconstruction.timing import TimerLog, status

VALIDATION_DIR = PROJECT_ROOT / "outputs" / "formal_validation"
TEST_DIR = PROJECT_ROOT / "outputs" / "formal_heldout_test"
LOIO_DIR = PROJECT_ROOT / "outputs" / "leave_one_individual_out"
TRANSFER_DIR = PROJECT_ROOT / "outputs" / "transfer_analysis"
FIG_DIR = PROJECT_ROOT / "outputs" / "publication_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

timer = TimerLog()

In [ ]:
import matplotlib.pyplot as plt

def read_csv_if_exists(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def save_bar(df, x, y, title, filename, rotate=True):
    if df is None or df.empty or x not in df.columns or y not in df.columns:
        print(f"Skipping {filename}: missing data")
        return
    fig, ax = plt.subplots(figsize=(max(7, 0.45 * len(df)), 4.5))
    ax.bar(df[x].astype(str), pd.to_numeric(df[y], errors="coerce"))
    ax.set_title(title)
    ax.set_ylabel(y)
    ax.set_xlabel(x)
    if rotate:
        ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=300)
    fig.savefig(FIG_DIR / filename.replace(".png", ".pdf"))
    plt.close(fig)

def keep_key_methods(df):
    if df is None or df.empty or "method" not in df.columns:
        return df
    key = [
        "linear_interpolation",
        "robust_pretrained_top1",
        "balanced_pooled_top10_candidate_set",
    ]
    return df[df["method"].astype(str).isin(key)].copy()

In [ ]:
status("Figures 1/4: loading summary tables")
val_summary = read_csv_if_exists(VALIDATION_DIR / "paper_formal_validation_method_summary.csv")
test_summary = read_csv_if_exists(TEST_DIR / "paper_formal_heldout_test_method_summary.csv")
loio_summary = read_csv_if_exists(LOIO_DIR / "paper_loio_method_summary.csv")
loio_individual = read_csv_if_exists(LOIO_DIR / "paper_loio_by_heldout_individual_method_summary.csv")
transfer_test = read_csv_if_exists(TRANSFER_DIR / "heldout_test_method_summary_by_best_train_transfer_relation.csv")

display(test_summary.head() if not test_summary.empty else pd.DataFrame())

In [ ]:
status("Figures 2/4: method summary bar plots")
save_bar(keep_key_methods(val_summary), "method", "ADE_median", "Formal validation median ADE", "fig01_validation_method_median_ADE.png")
save_bar(keep_key_methods(test_summary), "method", "ADE_median", "Formal held-out test median ADE", "fig02_test_method_median_ADE.png")
save_bar(keep_key_methods(loio_summary), "method", "ADE_median", "LOIO median ADE across held-out individuals", "fig03_loio_method_median_ADE.png")

In [ ]:
status("Figures 3/4: LOIO individual-level plot")
if not loio_individual.empty and {"loio_heldout_individual_key", "method", "ADE_median"}.issubset(loio_individual.columns):
    focus = loio_individual[loio_individual["method"].astype(str).isin(["linear_interpolation", "robust_pretrained_top1", "balanced_pooled_top10_candidate_set"])].copy()
    if not focus.empty:
        pivot = focus.pivot_table(index="loio_heldout_individual_key", columns="method", values="ADE_median", aggfunc="first")
        ax = pivot.plot(kind="bar", figsize=(max(8, 0.45 * len(pivot)), 5))
        ax.set_title("LOIO median ADE by held-out individual")
        ax.set_ylabel("Median ADE (m)")
        ax.set_xlabel("Held-out individual")
        fig = ax.get_figure()
        fig.tight_layout()
        fig.savefig(FIG_DIR / "fig04_loio_by_individual_median_ADE.png", dpi=300)
        fig.savefig(FIG_DIR / "fig04_loio_by_individual_median_ADE.pdf")
        plt.close(fig)

In [ ]:
status("Figures 4/4: transfer plot and master tables")
if not transfer_test.empty and {"best_train_transfer_relation", "method", "ADE_median"}.issubset(transfer_test.columns):
    focus = transfer_test[transfer_test["method"].astype(str).isin(["linear_interpolation", "robust_pretrained_top1", "balanced_pooled_top10_candidate_set"])].copy()
    if not focus.empty:
        pivot = focus.pivot_table(index="best_train_transfer_relation", columns="method", values="ADE_median", aggfunc="first")
        ax = pivot.plot(kind="bar", figsize=(9, 5))
        ax.set_title("Held-out test median ADE by transfer relation")
        ax.set_ylabel("Median ADE (m)")
        ax.set_xlabel("Best training support relation")
        fig = ax.get_figure()
        fig.tight_layout()
        fig.savefig(FIG_DIR / "fig05_transfer_relation_median_ADE.png", dpi=300)
        fig.savefig(FIG_DIR / "fig05_transfer_relation_median_ADE.pdf")
        plt.close(fig)

# Save a compact master workbook-like set of CSV copies.
for name, df in [
    ("table_validation_method_summary.csv", val_summary),
    ("table_heldout_test_method_summary.csv", test_summary),
    ("table_loio_method_summary.csv", loio_summary),
    ("table_transfer_test_summary.csv", transfer_test),
]:
    if df is not None and not df.empty:
        df.to_csv(FIG_DIR / name, index=False)

print("Saved figures/tables to:", FIG_DIR)